# Exploratory Analysis - India Air Quality & Weather Tracker

This notebook reads directly from `data/air_quality.db`, which is populated by:
1. `scripts/backfill_history.py` — run once, loads 30 days of hourly history per city
2. `scripts/fetch_latest.py` — run hourly via GitHub Actions, keeps adding new readings

Because the database keeps growing, re-running this notebook later will show more data and potentially different trends than the day it was first written.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/air_quality.db')
df = pd.read_sql('SELECT * FROM air_quality_readings', conn)
conn.close()

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
df = df.dropna(subset=['timestamp']).sort_values('timestamp')
print(df.shape)
df.head()

## Which city has the worst average air quality?

In [ ]:
avg_aqi = df.groupby('city')['us_aqi'].mean().sort_values(ascending=False)
avg_aqi

**Note:** run this yourself after the pipeline has collected real data — the ranking above will reflect actual conditions on the day you run it, not a fixed finding. That's the point of a live pipeline vs a static dataset.

## Does temperature correlate with PM2.5 levels?

In [ ]:
correlation = df[['temperature_c', 'pm2_5']].corr().iloc[0, 1]
print(f'Correlation between temperature and PM2.5: {correlation:.3f}')

## Hour-of-day pattern — does pollution spike at certain times?

In [ ]:
df['hour'] = df['timestamp'].dt.hour
hourly_avg = df.groupby('hour')['pm2_5'].mean()
hourly_avg.plot(kind='line', marker='o', figsize=(8,4), title='Average PM2.5 by Hour of Day')

## Summary
Update this section with your own findings once the pipeline has run for a few days — cite the specific numbers you see (e.g. "City X averaged Y AQI, Z% higher than City W") the same way the Startup Funding project's README does. That's what makes it defensible in an interview: the insight is something you actually observed, not a number I generated for you.